# Habitat 居家双任务：Baseline MPC 与元认知 MPC

本实验在 ReplicaCAD 居家场景中使用 Fetch 机器人完成两个独立任务：**抓取桌上的水杯**、**打开厨房抽屉**。导航是两个任务共同的前置过程。比较对象为固定规划预算的 model-based RL baseline，以及使用相同世界模型结构、任务目标和动作能力的元认知 MPC。

元认知模块先读取预测误差、搜索改善、碰撞和执行进展，再预测阶段可靠性并分配后续搜索计算。它区分监测分数、概率校准和控制选择；不会用“信心超过 85%”作为通用执行开关。世界模型的更新次数和 FIFO replay 规则保持固定。


**预算：**默认两个种子共 480 个训练 episode、4240 个验证/测试 episode，另有环境检查、每种子 240 个几何 reset 和 30 个共同轨迹监测诊断 episode。pilot 指每个模型的训练预算较小，完整比较仍有较多评估。若先检查流程，可在 Cell 5 将 `SEEDS` 设为 `[0]`；正式比较再使用多个种子。该入口已清除运行输出，实际运行记录与已导出的结果保存在仓库 `results/` 中。

## Cell 1 · 展开源码并设置路径

只需上传这个 notebook。下面的压缩内容已经包含独立 Python 源码、测试和中文说明，无需另外上传源码 ZIP。`code_meta_v2` 存源码，`outputs_meta_v2` 存实验结果；重新执行本单元会恢复内嵌源码。现有 Habitat 安装和数据可复用。

In [ ]:
import base64, io, json, os, subprocess, sys, zipfile
from pathlib import Path
from IPython.display import display, Image, Video

WORK = Path('/content/habitat_cup_demo')
CODE = WORK / 'code_meta_v2'
ENV = WORK / 'habitat-env'
PYTHON = ENV / 'bin/python'
DATA = WORK / 'data'
OUTPUT_ROOT = WORK / 'outputs_meta_v2'
CODE.mkdir(parents=True, exist_ok=True)
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
TASKS = ('pick_cup', 'open_drawer')
TASK_LABELS = {'pick_cup': '取杯', 'open_drawer': '开抽屉'}
DOMAIN_LABELS = ['A 原始环境', 'B 通道受阻', 'C 操作环境拥挤']
AGENTS = ('baseline', 'metacognitive')
PAYLOAD = '...'
with zipfile.ZipFile(io.BytesIO(base64.b64decode(PAYLOAD))) as archive:
    archive.extractall(CODE)
print('源码：', CODE)
print('环境：', ENV)
print('数据：', DATA)

## Cell 2 · 安装或复用 Habitat



In [ ]:
probe_code = (
    "import habitat_sim, habitat, torch, numpy; "
    "assert habitat_sim.__version__ == '0.3.3'; "
    "assert numpy.__version__ == '1.26.4'"
)
child_env = os.environ.copy()
child_env.pop('PYTHONPATH', None)
child_env.pop('PYTHONHOME', None)
child_env['MPLBACKEND'] = 'Agg'
ready = PYTHON.exists() and subprocess.run(
    [str(PYTHON), '-c', probe_code], env=child_env, capture_output=True
).returncode == 0
if ready:
    print('复用已安装的 Habitat 0.3.3 环境。')
else:
    subprocess.run([sys.executable, str(CODE / 'bootstrap.py'), '--prefix', str(ENV)],
                   env=child_env, check=True)
print('Habitat Python：', PYTHON)

## Cell 3 · 准备场景数据与运行入口

下载或验证 ReplicaCAD、Fetch 资产。`run_habitat` 在独立环境中执行每个命令，把完整输出保留为日志；控制台只省略重复插件提示。发生错误时检查最后的异常与日志路径，插件重复提示通常不是程序退出的原因。

In [ ]:
def run_habitat(arguments, tag='run'):
    """Run one Habitat subprocess and preserve its complete diagnostic log."""
    env = os.environ.copy()
    env.pop('PYTHONPATH', None)
    env.pop('PYTHONHOME', None)
    env.update(PYTHONUNBUFFERED='1', MPLBACKEND='Agg', HABITAT_SIM_LOG='error',
               MAGNUM_LOG='quiet', OMP_NUM_THREADS='1', OPENBLAS_NUM_THREADS='1')
    logs = WORK / 'logs_meta_v2'
    logs.mkdir(parents=True, exist_ok=True)
    log_path = logs / (tag + '.log')
    command = [str(PYTHON)] + [str(arg) for arg in arguments]
    print('运行：', ' '.join(command))
    with log_path.open('w') as log:
        proc = subprocess.Popen(command, cwd=str(CODE), env=env,
                                stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
        try:
            for line in proc.stdout:
                log.write(line)
                log.flush()
                if not line.startswith('PluginManager::Manager: duplicate'):
                    print(line, end='')
            result = proc.wait()
        except BaseException:
            proc.terminate()
            try:
                proc.wait(timeout=5)
            except subprocess.TimeoutExpired:
                proc.kill()
                proc.wait()
            raise
    if result:
        tail = '\n'.join(log_path.read_text(errors='replace').splitlines()[-60:])
        raise RuntimeError(f'进程退出码 {result}；日志：{log_path}\n最后 60 行：\n{tail}')
    return log_path

run_habitat(['download_assets.py', '--data', DATA], 'assets')

## Cell 4 · 检查代码与任务接口

先执行回归检查，再用**未训练的先验参考 MPC**在一个固定布局、固定种子下检查两任务 × A/B/C。这里验证环境与动作接口是否可工作。

`CHECK` 后两个数字为成功标记和步数。A 条件失败会停止；B/C 的策略失败会保留并显示，应结合导航成功、碰撞和抽屉开度判断困难在哪一段。参考控制器成功不能证明所有测试场景都可完成，参考控制器失败也不能证明学习后的模型一定失败。

成功标准：取杯后抬高至少 12 cm、保持 5 步；抽屉实际拉开至少 18 cm、保持 5 步。使用辅助抓握、运动学控制及新增障碍的几何碰撞近似，详见源码 `ENVIRONMENT_zh.md`。

In [ ]:
import pandas as pd
pd.set_option('display.max_columns', None)
pd.set_option('display.precision', 6)
run_habitat(['-m', 'unittest', 'discover', '-s', 'tests', '-v'], 'code_checks')
CHECK_DIR = WORK / 'outputs_meta_v2' / 'environment_check'
run_habitat(['-m', 'cup_baseline.run', 'smoke', '--data', DATA,
             '--out', CHECK_DIR], 'environment_check')
check = json.loads((CHECK_DIR / 'environment_check.json').read_text())
display(pd.DataFrame(check['rows'])[
    ['task', 'domain', 'success', 'steps', 'nav_success', 'collision_steps',
     'manipulation_collision_steps', 'drawer_open_m']
])
print(check['note'])

## Cell 5 · 统一设置 pilot、种子与结果位置

这里集中设置整套实验参数。`SEEDS` 是**独立训练种子**；每个种子下两组使用相同任务、布局与初始条件，实际动作轨迹允许不同。测试 episode 的 bootstrap 区间无法代替训练种子之间的变化，因此默认运行两个训练种子。

默认每组、每种子：基础训练 60 个 episode，两任务各 30 个；其中前 4 个先验采集 episode 计入这 60 个。每次基础验证每任务 10 个，最终 seen/unseen 每任务各 20 个。seen/unseen 是相同任务在训练布局/未见布局中的评估。

'smoke'    →   4 episodes：两任务各 2 次，用于快速检查流程。
'pilot'    →  60 episodes：两任务各 30 次，用于初步比较。
'research' → 300 episodes：两任务各 150 次，用于较大规模实验。
preset 还会同时调整验证/测试数量、模型更新次数和 MPC 搜索预算。

持续学习：每组每阶段 20 个训练 episode，两任务各 10 个；每 4 个总训练 episode 验证一次。基础训练与持续学习的模型更新、FIFO replay 和训练样本分别计数。



In [ ]:
# smoke: 4；pilot: 60；research: 300 个基础训练 episode / agent / seed。
# preset 同时调整模型更新、搜索和评估预算；EPISODES 仅覆盖训练 episode 数。
PRESET = 'pilot'
DYNAMICS = 'learned'
SEEDS = [0, 1]              # 与已提供的实验结果一致：两个独立训练种子。
EPISODES = None              # None 使用 preset；pilot 为每组每种子 60 个。
REPLAY_LIMIT = 4096          # 世界模型保存的 transition 上限，按 FIFO 淘汰。
RESUME = True
META_COMPUTE_WEIGHT = 0.10   # 训练前固定；不得根据正式测试结果调参。
META_STEP_WEIGHT = 0.05
PHASE_EPISODES = 20          # 每个 A/B/C 阶段的两任务合计训练 episode。
CL_EVAL_N = 20               # 每个任务、每个条件的保留测试 episode。
CL_VALIDATION_N = 10         # 每个任务的阶段适应验证 episode。
CL_CHECK_EVERY = 4
SHADOW_N = 5                 # 共同轨迹诊断：每任务/条件 5 个，共 30 个/seed。
GENERATION_VERSION = 'passage_v2'
USE_DRIVE = True            # 将结果和 checkpoint 保存在 Drive，便于重连后恢复。

if not SEEDS or len(SEEDS) != len(set(SEEDS)) or any(type(s) is not int or s < 0 for s in SEEDS):
    raise ValueError('SEEDS 必须是非空、不重复的非负整数列表。')
if EPISODES is not None and (EPISODES < 2 or EPISODES % 2):
    raise ValueError('EPISODES 必须是大于等于 2 的偶数。')
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    OUTPUT_ROOT = Path('/content/drive/MyDrive/HabitatHomeTwoTasks/outputs_meta_v2')
else:
    OUTPUT_ROOT = WORK / 'outputs_meta_v2'
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
EXPERIMENT_DIRS = {
    seed: OUTPUT_ROOT / f'meta_v2_{DYNAMICS}_{PRESET}_seed{seed}' for seed in SEEDS
}
RUN_DIRS_BY_SEED = {
    seed: {agent: experiment / agent for agent in AGENTS}
    for seed, experiment in EXPERIMENT_DIRS.items()
}
CL_DIRS_BY_SEED = {
    seed: {agent: experiment / f'continual_{agent}_{GENERATION_VERSION}' for agent in AGENTS}
    for seed, experiment in EXPERIMENT_DIRS.items()
}
AGGREGATE_DIR = OUTPUT_ROOT / f'aggregate_{DYNAMICS}_{PRESET}'
print('训练种子：', SEEDS)
print('实验结果：', OUTPUT_ROOT)
print('跨种子汇总：', AGGREGATE_DIR)

## Cell 6 · 每个种子训练两组模型

baseline 使用固定 CEM 规划预算；元认知在共享动作能力下分配搜索批次，并可回退到固定 MPC。世界模型为三个 MLP 的集成，学习真实动作后的状态变化；两组都使用 MPC 在这个模型中预测候选动作序列，然后执行第一个动作并重新规划。

两组世界模型使用相同结构、初始种子和更新规则，但各自收集轨迹并训练，训练后的权重可能不同。因此闭环比较衡量整个系统的效果；它不能把差异全部归因于监测头。Cell 8 另用共同轨迹诊断概率校准。

元认知的阶段成功、碰撞和预测误差监督来自实际训练反馈；概率校准使用训练内部保留数据。诊断头与 Q 头共享表示，调控由 Q 值选择；校准概率本身不作为执行阈值，改善校准不保证行为改变。调控只学习实际执行选项的结果，不为未执行选项伪造反馈。所有初始/追加搜索、回退、监测前向和训练更新分别计费。世界模型固定更新、4096 条 FIFO replay 保持一致；元认知额外的监督记录与校准记录单独报告。

验证/测试冻结两个模型的参数、校准参数和所有训练缓冲区。episode 内可以累积误差历史并调整计算，episode 结束后清空局部历史。冻结不是禁止 agent 思考或重新规划，而是防止测试结果变成训练数据。

In [ ]:
def train_agent(agent, seed, run_dir):
    command = ['-m', 'cup_baseline.run', 'train', '--data', DATA, '--out', run_dir,
               '--preset', PRESET, '--dynamics', DYNAMICS, '--seed', seed,
               '--agent', agent, '--replay-limit', REPLAY_LIMIT,
               '--meta-compute-weight', META_COMPUTE_WEIGHT,
               '--meta-step-weight', META_STEP_WEIGHT]
    if EPISODES is not None:
        command += ['--episodes', EPISODES]
    if RESUME and (run_dir / 'checkpoint.pt').exists():
        command += ['--resume']
    run_habitat(command, f'train_meta_v2_{agent}_seed{seed}')

for seed in SEEDS:
    for agent in AGENTS:
        print(f'基础训练：seed={seed}, agent={agent}')
        train_agent(agent, seed, RUN_DIRS_BY_SEED[seed][agent])
print('基础训练完成。下一单元执行必测的 A→B→C 持续学习。')

## Cell 7 · A→B→C 持续学习

从各组自己的基础 checkpoint 出发，依次训练 A 原始环境、B 通道受阻、C 操作环境拥挤。B 增加会导致绕行的纸箱；C 在杯子或抽屉把手的操作区域增加物体。B/C 各自相对 A 变化，不把两种变化累积叠加，也不改变动作增益。

每个种子先检查本协议全部不同初始环境。默认 240 次纯几何 reset 不执行策略、不学习、不计入训练样本；场景生成失败会停止并保存诊断，不记作 agent 失败，也不根据策略成功率筛选环境。B 若必须重新采样起点，会记录该变化；两组使用匹配几何。几何检查通过并不证明机械臂任意姿态都可达。

每组、每种子默认 60 个持续学习训练 episode、840 个评估 episode。阶段内以冻结验证曲线衡量适应；阶段结束冻结评估全部 A/B/C，形成保留矩阵；推理与更新代价分别记录。世界模型、元认知模型和 replay 在训练 episode 间持续保留。

In [ ]:
def check_continual_geometry(seed):
    run_habitat(['-m', 'cup_baseline.run', 'check_domains', '--data', DATA,
                 '--out', EXPERIMENT_DIRS[seed] / f'generation_check_{GENERATION_VERSION}',
                 '--checkpoint', RUN_DIRS_BY_SEED[seed]['baseline'] / 'checkpoint.pt',
                 '--phase-episodes', PHASE_EPISODES, '--eval-n', CL_EVAL_N,
                 '--validation-n', CL_VALIDATION_N, '--check-every', CL_CHECK_EVERY],
                f'geometry_meta_v2_seed{seed}_{GENERATION_VERSION}')

def run_continual(agent, seed):
    run_habitat(['-m', 'cup_baseline.run', 'continual', '--data', DATA,
                 '--out', CL_DIRS_BY_SEED[seed][agent],
                 '--checkpoint', RUN_DIRS_BY_SEED[seed][agent] / 'checkpoint.pt',
                 '--phase-episodes', PHASE_EPISODES, '--eval-n', CL_EVAL_N,
                 '--validation-n', CL_VALIDATION_N, '--check-every', CL_CHECK_EVERY],
                f'continual_meta_v2_{agent}_seed{seed}_{GENERATION_VERSION}')

for seed in SEEDS:
    check_continual_geometry(seed)
    for agent in AGENTS:
        print(f'持续学习：seed={seed}, agent={agent}')
        run_continual(agent, seed)
print('所有指定种子的两组持续学习已完成。')

## Cell 8 · 分种子比较与跨种子汇总

生成可追溯的 CSV、JSON 和 PNG，不预设哪组获胜。每种子的两组配置、测试初始条件与持续学习几何必须匹配；只汇总 Cell 5 明确指定且已完成的种子。

- **样本效率 N85**：验证成功率首次达到 85%，且下一检查点仍达标；未达标显示 `>预算`。单位为两任务合计的训练 episode。N85=0 表示初始验证已经达标。未达标值不能替换成预算或当作零参与普通平均。
- **计算代价**：NN FLOPs、批量/样本前向次数、实际环境步数。FLOPs 是计算代理量，非实测电耗；不包含物理、渲染、IK、几何与所有软件开销。每成功成本包含失败尝试的计算。
- **泛化**：分别看 seen/unseen 成功率；gap=seen−unseen。负值表示这批 unseen 布局表现更高；gap 接近零不自动等于泛化更好。
- **持续学习**：适应 N85/AUC、阶段后表现、保留矩阵、BWT/遗忘与阶段成本。冻结初始 SR 是整个持续学习开始前的基础模型；它不是每个阶段开始前的表现。沿同一矩阵列向下看才能观察变化。
- **不确定性**：单 seed 的配对 episode bootstrap 只反映该训练模型下的测试抽样；跨 seed 表按训练种子统计均值、样本标准差和数量。三个 seed 仍属于小规模实验，不把测试 episode 当独立训练重复。

本单元还先做**共同轨迹监测诊断**：固定 baseline 世界模型/规划器执行，从基础训练 checkpoint 读取的冻结元监测器只观察，不改变动作或学习。每种子默认 30 个 episode，报告同一轨迹上的原始/校准概率质量及独立成本；这些不计训练样本，也不并入下面基础/持续学习的成本表。该诊断读取 baseline 策略的状态分布，不能直接证明元认知闭环策略的反事实收益。

监测预测质量与任务表现分别保存。概率校准不保证控制有效；Brier/ECE 也不等于人类实验中的内在元认知噪声。应结合搜索选择、误差与执行轨迹分析机制。

In [ ]:
def comparison_command(seed):
    run_dirs = RUN_DIRS_BY_SEED[seed]
    cl_dirs = CL_DIRS_BY_SEED[seed]
    return ['-m', 'cup_baseline.compare', '--baseline', run_dirs['baseline'],
            '--metacognitive', run_dirs['metacognitive'],
            '--baseline-cl', cl_dirs['baseline'], '--metacognitive-cl', cl_dirs['metacognitive'],
            '--out', EXPERIMENT_DIRS[seed] / 'comparison']

for seed in SEEDS:
    shadow_dir = EXPERIMENT_DIRS[seed] / 'shadow'
    run_habitat(['-m', 'cup_baseline.run', 'shadow', '--data', DATA,
                 '--out', shadow_dir,
                 '--checkpoint', RUN_DIRS_BY_SEED[seed]['baseline'] / 'checkpoint.pt',
                 '--monitor-checkpoint', RUN_DIRS_BY_SEED[seed]['metacognitive'] / 'checkpoint.pt',
                 '--n', SHADOW_N], f'shadow_meta_v2_seed{seed}')
    shadow = json.loads((shadow_dir / 'shadow_summary.json').read_text())
    print(f'共同轨迹监测诊断：seed={seed}；以下成本单独报告。')
    display(pd.DataFrame([shadow['costs']]))
    display(pd.read_csv(shadow_dir / 'shadow_metrics.csv'))
    run_habitat(comparison_command(seed), f'compare_meta_v2_seed{seed}')
    folder = EXPERIMENT_DIRS[seed] / 'comparison'
    report = json.loads((folder / 'comparison.json').read_text())
    print(f'\n种子 {seed}：基础指标')
    display(pd.DataFrame(report['metrics']))
    print('配对成功率差：metacognitive − baseline；单位为百分点。')
    display(pd.DataFrame(report['paired']))
    for agent, continual in report['continual'].items():
        for task in TASKS:
            print(f'{agent} · {TASK_LABELS[task]}：行=学完阶段，列=测试条件，单位=%')
            display(pd.DataFrame(continual['matrix'][task], index=DOMAIN_LABELS,
                                 columns=DOMAIN_LABELS).mul(100).round(1))
    for name in ['calibration.csv', 'local_monitoring.csv', 'continual_adaptation.csv',
                 'continual_retention.csv', 'continual_costs.csv', 'continual_paired.csv']:
        print(name)
        display(pd.read_csv(folder / name))
    display(Image(filename=str(folder / 'comparison.png')))

run_habitat(['-m', 'cup_baseline.aggregate', '--experiments',
             *[EXPERIMENT_DIRS[seed] for seed in SEEDS], '--out', AGGREGATE_DIR],
            'aggregate_meta_v2')
for name in ['aggregate_metrics.csv', 'aggregate_paired.csv', 'aggregate_n85.csv',
             'aggregate_continual.csv', 'aggregate_costs.csv', 'aggregate_calibration.csv',
             'aggregate_local_monitoring.csv']:
    print(name)
    display(pd.read_csv(AGGREGATE_DIR / name))
display(Image(filename=str(AGGREGATE_DIR / 'aggregate.png')))
print('原始记录与单种子比较：', list(EXPERIMENT_DIRS.values()))
print('跨种子 CSV/JSON/PNG：', AGGREGATE_DIR)

## 方法依据与解释边界

Guggenmos（2022）区分决策、元认知读出和信心转换，为“监测证据→分数→校准概率”的可诊断结构提供启发；Fleming & Daw（2017）讨论对自身决策过程的二阶推断。Desender 等（2018）研究信心与信息搜集的关系；Hamrick 等（2017）研究内部模拟的资源分配；van den Berg 等（2016）研究作出初始决定后的证据和改变决定。这里据此提出机器人监测/控制的工程实现，**不是这些论文的严格复现，也不主张拟合出了人类元认知噪声参数**。

详细结构、训练标签、校准隔离、计费边界与验证方法见源码 `METACOGNITION_zh.md`、`LOCAL_VALIDATION_zh.md`。环境采用结构化几何观测与辅助抓握，不是 RGB 视觉策略、真实力控或跨机器人泛化基准。本 notebook 只在完成真实实验后生成结果；本地测试替身的数字不作为 Habitat 成绩。

文献：[Guggenmos, 2022](https://elifesciences.org/articles/75420) · [Fleming & Daw, 2017](https://pubmed.ncbi.nlm.nih.gov/28004960/) · [Desender et al., 2018](https://pubmed.ncbi.nlm.nih.gov/29608411/) · [Hamrick et al., 2017](https://arxiv.org/abs/1705.02670) · [van den Berg et al., 2016](https://elifesciences.org/articles/12192)

## Cell 9 · 可选：录制实际执行视频（最后运行）

所有训练和指标完成后，再选择一个训练种子、一个 agent 和一个条件录像。默认使用其基础训练 checkpoint；这是一条定性示例，不代表平均成功率，不参与正式统计。本单元不重新训练模型。

如需录像，在 Colab 选择带 NVIDIA GPU 的运行时，并确认当前运行时能看到 GPU。切换运行时可能清空临时磁盘，先确保 checkpoint 已保存。脚本先检查实际图像帧是否可读，再录制；渲染失败保留诊断日志，不影响已完成的实验结果。

In [ ]:
RECORD_VIDEO = False
VIDEO_SEED = None             # None 表示选择 SEEDS 中的第一个种子；也可填 0 或 1。
VIDEO_AGENT = 'metacognitive'  # 'baseline' 或 'metacognitive'。
VIDEO_DOMAIN = 0              # 0=A 原始环境，1=B 通道受阻，2=C 操作环境拥挤。
if RECORD_VIDEO:
    required = ('SEEDS', 'RUN_DIRS_BY_SEED', 'AGENTS', 'TASKS',
                'CODE', 'PYTHON', 'DATA', 'WORK')
    missing = [name for name in required if name not in globals()]
    if missing:
        raise RuntimeError(
            '当前会话缺少配置：' + ', '.join(missing)
            + '。请先按顺序执行 Cell 1–5 恢复环境、路径与配置，并确认 checkpoint 已保存。')
    selected_video_seed = SEEDS[0] if VIDEO_SEED is None else VIDEO_SEED
    if selected_video_seed not in RUN_DIRS_BY_SEED or VIDEO_AGENT not in AGENTS:
        raise ValueError('请选择本次实验中已训练的 seed 和 agent。')
    if VIDEO_DOMAIN not in (0, 1, 2):
        raise ValueError('VIDEO_DOMAIN 必须为 0、1 或 2。')
    import runpy
    from IPython.display import Video, display
    video_tools = runpy.run_path(str(CODE / 'record_video.py'))
    for task in TASKS:
        try:
            video_path = video_tools['repair_and_record'](
                CODE, PYTHON, DATA, RUN_DIRS_BY_SEED[selected_video_seed][VIDEO_AGENT],
                WORK, task=task, domain=VIDEO_DOMAIN)
            display(Video(str(video_path), embed=True))
        except Exception as error:
            print('录像未完成：', error)
            print('请检查渲染诊断；训练、持续学习和汇总结果已保留。')
else:
    print('已跳过可选录像。')
    if 'OUTPUT_ROOT' in globals():
        print('实验结果目录：', OUTPUT_ROOT)
